In [9]:
# Edward

REPAIR_TRIGGER = 0.5 # 50% of new battery capacity (degradation).

PROFIT_MARGIN_HIGH = 0.05 # 5% above/below average price.
PROFIT_MARGIN_LOW = 1 - (1 / (1 + PROFIT_MARGIN_HIGH))

PROFIT_SCALE_UP = 1.5 # 50% more profit margin.
PROFIT_SCALE_DOWN = (1 / PROFIT_SCALE_UP) # 33% less profit margin.

LOW_CHARGE = 0.2 # 20% charge.
HIGH_CHARGE = 0.8 # 80% charge.
VERY_LOW_CHARGE = 0.05 # 5% charge.
VERY_HIGH_CHARGE = 0.95 # 95% charge.

# This baseline buys low and sells high, but also tries to avoid <20% or >80% charge to reduce battery degradation.
# price: current electricity price.
# average_price: the moving average of the electricity price.
# battery_charge: "percentage" of current battery charge level.
# battery_degradation: "percentage" of current battery capacity divided by new battery capacity.
def rule_based_baseline(price, moving_average_price, battery_charge, battery_degradation):
    if (battery_degradation < REPAIR_TRIGGER):
        return 3 # Replace battery.
    elif (battery_charge < VERY_LOW_CHARGE):
        # Slightly undercharged.
        if (price < moving_average_price * (1 - PROFIT_MARGIN_LOW * PROFIT_SCALE_DOWN)):
            return 0 # Buy with less profit.
        elif (price > moving_average_price * (1 - PROFIT_MARGIN_HIGH * PROFIT_SCALE_UP * PROFIT_SCALE_UP)):
            return 1 # Sell with a lot more profit.
        else:
            return 2 # Hold.
    elif (battery_charge > VERY_HIGH_CHARGE):
        # Slightly overcharged.
        if (price < moving_average_price * (1 - PROFIT_MARGIN_LOW * PROFIT_SCALE_UP * PROFIT_SCALE_UP)):
            return 0 # Buy with a lot more profit.
        elif (price > moving_average_price * (1 - PROFIT_MARGIN_HIGH * PROFIT_SCALE_DOWN)):
            return 1 # Sell with less profit.
        else:
            return 2 # Hold.
    elif (battery_charge < LOW_CHARGE):
        # Slightly undercharged.
        if (price < moving_average_price * (1 - PROFIT_MARGIN_LOW)):
            return 0 # Buy.
        elif (price > moving_average_price * (1 - PROFIT_MARGIN_HIGH * PROFIT_SCALE_UP)):
            return 1 # Sell with more profit.
        else:
            return 2 # Hold.
    elif (battery_charge > HIGH_CHARGE):
        # Slightly overcharged.
        if (price < moving_average_price * (1 - PROFIT_MARGIN_LOW * PROFIT_SCALE_UP)):
            return 0 # Buy with more profit.
        elif (price > moving_average_price * (1 + PROFIT_MARGIN_HIGH)):
            return 1 # Sell.
        else:
            return 2 # Hold.
    else:
        # Normally charged.
        if (price < moving_average_price * (1 - PROFIT_MARGIN_LOW)):
            return 0 # Buy.
        elif (price > moving_average_price * (1 + PROFIT_MARGIN_HIGH)):
            return 1 # Sell.
        else:
            return 2 # Hold.

# Test
print(rule_based_baseline(100, 110, 0.5, 0.9)) # Buy.
print(rule_based_baseline(110, 100, 0.5, 0.9)) # Sell.
print(rule_based_baseline(100, 110, 0.99, 0.9)) # Hold.
print(rule_based_baseline(100, 110, 0.5, 0.49)) # Repair.

0
1
2
3
